In [ ]:
# standard imports
import numpy as np
import matplotlib.pyplot as plt

import xarray as xr

# import to load ARGO float data
from argopy import DataFetcher as ArgoDataFetcher

from dask.diagnostics import ProgressBar

In [3]:
### Import Sea Surface Temperature (SST) data from NOAA

# Define the OPeNDAP URL for the NOAA OISST dataset
sst_url = "https://coastwatch.pfeg.noaa.gov/erddap/griddap/ncdcOisst21Agg_LonPM180"

# Open the dataset with xarray
ds_sst = xr.open_dataset(sst_url, chunks='auto')

# Subset the data to specific region and time of interest, download subset to memory
# This is Gulf Stream region in Jan 2023
subset_sst = ds_sst.sel(time=slice('2023-01-01', '2023-01-31'), latitude=slice(25.0, 45.0), longitude=slice(-80.0, -50.0))

# Plot a quick check with subsampled resolution to save time
plt.clf()
subset_sst.sst.isel(time=0).plot(cmap='coolwarm')
plt.show()

curl error details: 


OSError: [Errno -68] NetCDF: I/O failure: 'https://coastwatch.pfeg.noaa.gov/erddap/griddap/ncdcOisst21Agg_LonPM180'

In [ ]:
### Import ARGO Float data

# Spatial-temporal bounds format: [lon_min, lon_max, lat_min, lat_max, depth_min, depth_max, date_start, date_end]
argo_fetcher = ArgoDataFetcher().region([-80, -50, 25, 45, 0, 1000, '2023-01-01', '2023-01-31'])

# Fetch the data and convert to an xarray dataset
ds_argo = argo_fetcher.to_xarray()

# Argo data is initially loaded as a 1D list. Convert to standard depth-profile grid for easier analysis
ds_argo_profiles = ds_argo.argo.point2profile()

# Plot a quick temperature-depth profile for the first float
ds_argo_profiles.isel(N_PROF=0).plot.scatter(x="TEMP", y="PRES", yincrease=False, edgecolors='none')

In [ ]:
# Calculate the spatial and temporal bounding box of Argo profiles
min_lon, max_lon = ds_argo_profiles['LONGITUDE'].min().values, ds_argo_profiles['LONGITUDE'].max().values
min_lat, max_lat = ds_argo_profiles['LATITUDE'].min().values, ds_argo_profiles['LATITUDE'].max().values
min_time, max_time = ds_argo_profiles['TIME'].min().values, ds_argo_profiles['TIME'].max().values

# Slice the spatial bounding box, then subsample by taking every 10th pixel
regional_sst = ds_sst.sel(time=slice(min_time, max_time), latitude=slice(min_lat, max_lat), longitude=slice(min_lon, max_lon)).isel(
    latitude=slice(None, None, 10), longitude=slice(None, None, 10))

In [ ]:
# Download regional block into local memory
from dask.diagnostics import ProgressBar

with ProgressBar():
    regional_sst_loaded = regional_sst.compute()


In [ ]:
# get SST data from the ARGO locations
colocated_sst = regional_sst_loaded.sst.sel(time=ds_argo_profiles['TIME'], latitude=ds_argo_profiles['LATITUDE'],
                                                     longitude=ds_argo_profiles['LONGITUDE'], method='nearest')

# Add colocated SST back into Argo dataset as a new variable
ds_argo_profiles['matched_sst'] = colocated_sst

# Convert the matched Argo/SST data into a pandas dataframe (GS_0123 means gulf stream january 2023)
df_GS_0123 = ds_argo_profiles.to_dataframe()

df_GS_0123.head()

In [ ]:
df_GS_0123.info()

- There a null values that we will have to deal with in the columns: PRES, PRES_ERROR, PSAL, PSAL_ERROR, TEMP, TEMP_ERROR
- There are a few non-numeric columns: DATA_MODE and DIRECTION are strings, and TIME and tim are in datetime64